# Job 50007138 - L*(24, ff=256), the last cell of the L* grid

**Headline.** At N=24, narrowing `dim_feedforward` from 1024 to 256 costs **nothing**:
`L*(24, ff=256) = 3 = L*(24, ff=1024)`, a width factor of **1.00x**, and on the shared
1500-instance pool the thin L=3 scores **0.9487 against the wide L=3's 0.9500**
(delta -0.0013, 50 won / 52 lost, McNemar exact **p = 0.92**) using **0.52x the parameters**
(1,259,042 vs 2,440,994).

This completes the `L*` vs `(N, ff)` grid the multi-step plan asked for:

| N | L* at ff=1024 | L* at ff=256 | width factor |
|---|---|---|---|
| 24 | 3 | **3** | **1.00x** |
| 48 | 4 | 6 | 1.50x |
| 96 | 8 | > 12 | > 1.50x |

**The MLP-width cost on reasoning depth is a curve in N, not a constant.** It is zero at N=24,
1.5x at N=48, and at N=96 the thin column does not clear the gate at any depth tried.

**The negative result in the pre-registration.** (b) predicted the thin L=3 would still be
*resolvably worse* than the wide L=3 on the pool. It is not - it is a tie at p=0.92. That was
the one claim with real information in it, and it was falsified in the pre-registered direction.

**Two matched-parameter pairs, both won by thin-and-deep.** The N=24 ladder now contains two
pairs of cells within 0.2% of each other in parameter count, and the thin-deep member wins both
(section 1). This replicates finding 13 on a pool it was never measured on.

**Everything here is a DECODING score** on the leaked-order task: `shuffle_adj=False`, so each
node's adjacency row still arrives sorted by the neighbour's true colour (finding 16). That is
deliberate - this run extends one ladder rather than starting a second.

Deck: `notebooks/sudoku_n24_ff256_2026-10-28/n24_ff256.pdf` (9 pages, built by
`python -m notebooks.make_n24_ff256_deck`). Provenance in section 5.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

SCRATCH = os.environ["SCRATCH"]
RUN = f"{SCRATCH}/spin_gnn/sudoku_n24_ff256_2026-10-28"      # this job: L=2, L=3 at ff=256
WIDE = f"{SCRATCH}/spin_gnn/sudoku_layer_scan_2026-10-24_n24"  # finding 11, the ff=1024 ladder
THIN = f"{SCRATCH}/spin_gnn/sudoku_ff_depth_tradeoff_2026-10-25_n24"  # finding 13, L=4 and L=6
pd.set_option("display.width", 200)

GATE, RHO = 0.90, 0.60
# (label, checkpoint substring, colour, trained by THIS job)
CELLS = [("L=2, ff=1024", "L02_ff01024", "0.45", False),
         ("L=3, ff=1024", "L03_ff01024", "#2ca02c", False),
         ("L=2, ff=256", "L02_ff00256", "#d62728", True),
         ("L=3, ff=256", "L03_ff00256", "#1f77b4", True),
         ("L=4, ff=256", "L04_ff00256", "#ff7f0e", False),
         ("L=6, ff=256", "L06_ff00256", "#9467bd", False)]
THIN_LADDER = ["L02_ff00256", "L03_ff00256", "L04_ff00256", "L06_ff00256"]

# All three depth evals used --rho 0.75 0.60 --n-pool 600 --min-per-bin 40 off the same seeded
# generator stream at N=24, so the bins line up instance for instance across every column.
depth = pd.concat([pd.read_csv(f"{WIDE}/depth_by_layers.csv"),
                   pd.read_csv(f"{THIN}/depth_by_ff.csv"),
                   pd.read_csv(f"{RUN}/depth_n24_ff256.csv")], ignore_index=True)
depth["k"] = (depth.solve_rate * depth.n).round().astype(int)
pool = pd.read_csv(f"{RUN}/sharedpool_n1500.csv")
sweep = pd.concat([pd.read_csv(f"{WIDE}/sweep.csv"), pd.read_csv(f"{THIN}/sweep.csv"),
                   pd.read_csv(f"{RUN}/sweep.csv")],
                  ignore_index=True).set_index(["num_layers", "dim_feedforward"])


def bins(tag, rho=RHO):
    return depth[depth.checkpoint.str.contains(tag)].query("rho_requested == @rho").sort_values("T")


def cleared(tag):
    g = bins(tag)
    return int((g.solve_rate >= GATE).sum()), len(g)


def pooled_fisher(tag1, tag2):
    """Pooled solve_rate of two columns over the same bins, Fisher exact.

    The bins are shared instance for instance, so Fisher on the pooled 2x2 throws away the
    pairing and is therefore conservative. agg_eval's McNemar is the paired test, and it only
    covers the five checkpoints that went into the shared pool."""
    g1, g2 = bins(tag1), bins(tag2)
    a, b = int(g1.k.sum()), int(g1.n.sum() - g1.k.sum())
    c, d = int(g2.k.sum()), int(g2.n.sum() - g2.k.sum())
    return a / (a + b), c / (c + d), stats.fisher_exact([[a, b], [c, d]])[1]


def lstar_thin():
    """Smallest num_layers in the thin column clearing every rho=0.60 bin."""
    for tag in THIN_LADDER:
        c, n = cleared(tag)
        if n and c == n:
            return int(tag[1:3])
    return None


def ca_trend(T, k, n):
    """Cochran-Armitage trend across ordered depth bins (as notebooks.make_n96_lstar_deck)."""
    T, k, n = np.asarray(T, float), np.asarray(k, float), np.asarray(n, float)
    N, K = n.sum(), k.sum()
    p = K / N
    Tbar = (n * T).sum() / N
    z = (k * (T - Tbar)).sum() / np.sqrt(p * (1 - p) * (n * (T - Tbar) ** 2).sum())
    return z, 2 * stats.norm.sf(abs(z))


print(len(pool), "pool rows;", len(depth), "depth rows;", len(sweep), "sweep rows")
print("L*(24, ff=256) =", lstar_thin(), "| L*(24, ff=1024) = 3 (finding 11)")

## 1. Metric of record: one pool, five checkpoints

`sudoku_agg_eval --n-instances 1500 --seed-start 50000000000` builds a single instance pool far
above every training seed and scores all five checkpoints on it. The wide `L=3` is passed first,
so `vs_reference` and `delta` read against `L*(24, ff=1024)`. All five share `N=24`, `degree=5`,
`n_colors=4`, `shuffle_adj=False` and the same `INSTANCE_KEYS`, so the pool is poolable and
McNemar's exact test is the right paired test.

In [ ]:
POOL_ORDER = ["L=3, ff=1024 (ref)", "L=3, ff=256 (new)", "L=2, ff=256 (new)",
              "L=4, ff=256", "L=6, ff=256"]
pool = pool.assign(cell=POOL_ORDER)
display(pool[["cell", "solve_rate", "wilson_lo", "wilson_hi", "delta", "discordant_won",
              "discordant_lost", "mcnemar_exact_p", "truncated_rate",
              "outcome_accuracy"]].round(4))

# Architecture and teacher-forced loss, pulled per cell so nothing here is hand-typed.
ref_params = sweep.loc[(3, 1024)].n_params
rows = []
for name, tag, _, new in CELLS:
    L, ff = int(tag[1:3]), int(tag.split("ff")[1])
    r = sweep.loc[(L, ff)]
    pr = pool[pool.checkpoint.str.contains(tag)]
    rows.append({"cell": name, "new": new, "n_params": int(r.n_params),
                 "vs_wide_knee": round(r.n_params / ref_params, 3),
                 "val_nll": round(r.final_val_nll, 4),
                 "val_acc": round(r.final_val_accuracy, 4),
                 "pool_solve_rate": round(pr.solve_rate.iloc[0], 4) if len(pr) else None})
arch = pd.DataFrame(rows)
display(arch)

z = np.load(f"{RUN}/sharedpool_n1500_per_instance.npz", allow_pickle=True)
s = z["solved"]
print("solved per checkpoint:", s.sum(1), "of", s.shape[1], "| seed_start", z["seed_start"])
w, l = int((s[4] & ~s[1]).sum()), int((~s[4] & s[1]).sum())
print(f"L=6/ff=256 vs L=3/ff=256 (both thin): {w} won / {l} lost, "
      f"McNemar exact p={stats.binomtest(w, w + l, 0.5).pvalue:.2g}")

### Two matched-parameter pairs, both won by thin-and-deep

The N=24 ladder happens to contain two pairs of cells within **0.2%** of each other in parameter
count, one wide-and-shallow and one thin-and-deep. The thin-deep member wins both, which is
finding 13 replicated on evals it was never measured on:

| params | wide-shallow | thin-deep | gap | test |
|---|---|---|---|---|
| ~1.65M | L=2, ff=1024: 0.385 | **L=4, ff=256: 0.980** | +0.595 | Fisher p=1.1e-116 (bins) |
| ~2.44M | L=3, ff=1024: 0.969 | **L=6, ff=256: 0.995** | +0.026 | Fisher p=0.0024 (bins) |
| ~2.44M | L=3, ff=1024: 0.9500 | **L=6, ff=256: 0.9767** | +0.0267 | **McNemar p=3.0e-05 (pool)** |

The third row is the paired version of the second on the 1500-instance pool, so the 2.44M pair is
confirmed by the metric of record and not only by the depth bins.

**It is not a parameter story and it is not a width story - it is a depth story.** Starting from
the thin `L=2` (0.86M), spending 0.79M parameters on MLP width (-> `L=2/ff=1024`, 1.65M) buys
+0.095 on the bins. Spending the *same* 0.79M on layers at constant width
(-> `L=4/ff=256`, 1.65M) buys +0.690, Fisher p=8.9e-147. Same budget, same starting point, 7x the
return.

**The thin L=6 is also the best-fitting model at N=24 outright.** Its `final_val_nll` of 0.947 is
the lowest of all thirteen N=24 cells trained so far, below the wide L=4's 0.988 and the wide
L=8's 1.010 at 2.6x its parameter count.

In [ ]:
for lab, wide_tag, thin_tag in [("~1.65M", "L02_ff01024", "L04_ff00256"),
                                ("~2.44M", "L03_ff01024", "L06_ff00256")]:
    pt, pw, p = pooled_fisher(thin_tag, wide_tag)
    nw = int(sweep.loc[(int(wide_tag[1:3]), int(wide_tag.split("ff")[1]))].n_params)
    nt = int(sweep.loc[(int(thin_tag[1:3]), int(thin_tag.split("ff")[1]))].n_params)
    print(f"{lab}: {wide_tag} {nw:,} -> {pw:.4f} | {thin_tag} {nt:,} -> {pt:.4f} "
          f"| gap {pt - pw:+.4f}, Fisher p={p:.2g} | param gap {abs(nt - nw) / nw:.3%}")

pt, pw, p = pooled_fisher("L02_ff00256", "L02_ff01024")
print(f"\nwidth at L=2 (0.86M -> 1.65M): {pt:.4f} -> {pw:.4f}, {pw - pt:+.4f}, Fisher p={p:.2g}")
pt4, pt2, p4 = pooled_fisher("L04_ff00256", "L02_ff00256")
print(f"depth at ff=256 (0.86M -> 1.65M): {pt2:.4f} -> {pt4:.4f}, {pt4 - pt2:+.4f}, "
      f"Fisher p={p4:.2g}")
print("\nlowest final_val_nll at N=24:")
display(sweep.sort_values("final_val_nll")[["n_params", "final_val_nll",
                                            "final_val_accuracy"]].head(4).round(4))

## 2. Four depth bins, and where L* is read off

`L*` is the smallest `num_layers` reaching 0.90 on depth-`T` instances at fixed **realized**
density. `sudoku_depth_eval` bins a 600-instance pool by DD depth `T`; at `rho=0.60` the realized
density is 0.583-0.584 in every bin and every column, so depth varies and density does not.

N=24 only supports four bins (T=2..5) against N=96's six, and at this size most bins sit near the
ceiling, so the gate resolves little more than "clears" and "does not".

In [ ]:
for rho in (0.60, 0.75):
    g0 = bins("L03_ff01024", rho)
    if not len(g0):
        continue
    t = pd.DataFrame({name: bins(tag, rho).set_index("T").solve_rate.round(3)
                      for name, tag, _, _ in CELLS})
    t["n"] = g0.set_index("T").n
    print(f"=== rho={rho}, realized density {sorted(g0.realized_rho.round(4).unique())}")
    display(t)

for name, tag, _, _ in CELLS:
    g = bins(tag)
    zc, p = ca_trend(g["T"], g.k, g.n)
    c, n = cleared(tag)
    print(f"{name:<14} bins >= {GATE}: {c} of {n} | Cochran-Armitage z={zc:+.2f}, p={p:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.6))
for name, tag, col, new in CELLS:
    g = bins(tag)
    lo, hi = [], []
    for _, r in g.iterrows():
        a, b = stats.beta.interval(0.95, r.k + 0.5, r.n - r.k + 0.5)
        lo.append(max(0.0, r.solve_rate - a))   # a bin at exactly 1.0 would give a negative yerr
        hi.append(max(0.0, b - r.solve_rate))
    ax.errorbar(g["T"], g.solve_rate, yerr=[lo, hi], fmt="-o" if new else "--o", color=col,
                ms=6 if new else 4, lw=2.2 if new else 1.4, capsize=2,
                label=name + (" (new)" if new else ""))
ax.axhline(GATE, color="0.35", ls=":", lw=1.4)
ax.set_xlabel("DD depth T"); ax.set_ylabel("solve_rate")
ax.set_title("N=24, rho=0.60 (realized 0.583-0.584 in every bin); dotted line is the 0.90 gate")
ax.set_xticks([2, 3, 4, 5]); ax.set_ylim(0.0, 1.03)
ax.legend(fontsize=9, loc="center left", ncol=2)
fig.tight_layout()

**The thin column separates into two regimes with nothing in between.** `L=2/ff=256` sits at
0.28-0.31 across all four bins - flat, nowhere near the gate, and *below* the wide `L=2`'s
0.36-0.45. From `L=3` upward every thin cell clears every bin: 0.951-0.978 at L=3, 0.959-1.000 at
L=4, 0.990-1.000 at L=6. There is no cell at N=24 that clears some bins and misses others, which
is why the gate reading here is "3" with no ambiguity and also why it carries little resolution.

**MLP width is worth 9.5 points below the knee and zero at or above it.** Pooled over the four
bins the thin `L=2` scores 0.290 against the wide `L=2`'s 0.385 (Fisher p=0.001) - so at `L=2`
width genuinely helps. At `L=3` the same comparison is 0.965 vs 0.969 (p=0.87), a tie. Width buys
something only while the stack is too shallow to deduce at all, and stops mattering the moment it
is deep enough.

**No cell at N=24 declines with T the way N=96's did.** The Cochran-Armitage trends are small and
mostly insignificant at this size: with the deepest bin at T=5 and solve rates pinned near 1.0,
there is no room for the capacity-limited decline that finding 25 measured at N=96.

## 3. Mechanism: the N=96 grammar collapse does not travel down to N=24

`well_formed_rate` is the fraction of generations that parse as a trace at all. At N=96, finding
25's deep-thin cell (`L=12/ff=256`) failed *generatively*: its well-formedness collapsed to 0.853
while its deduction conditional on parsing stayed near the wide knee's. Pre-registration (d) asked
whether the same thing happens at N=24 - because if it did, the gate would be measuring output
grammar rather than reasoning, and every `L*` in the grid would need re-reading.

In [ ]:
rows = []
for name, tag, _, _ in CELLS:
    g = bins(tag)
    cond = g.solve_rate / g.well_formed_rate
    rows.append({"cell": name,
                 "worst_well_formed": round(g.well_formed_rate.min(), 4),
                 "best_well_formed": round(g.well_formed_rate.max(), 4),
                 "solve|wf T=2": round(cond.iloc[0], 3),
                 "solve|wf T=5": round(cond.iloc[-1], 3),
                 "truncated_rate": round(g.truncated_rate.max(), 4)})
display(pd.DataFrame(rows))

thin_bins = pd.concat([bins(t) for t in THIN_LADDER if len(bins(t))])
worst = thin_bins.well_formed_rate.min()
print(f"worst well_formed_rate anywhere in the N=24 thin column: {worst:.4f} "
      f"over all {len(thin_bins)} bins -> (d) {'CONFIRMED' if worst >= 0.95 else 'FALSIFIED'}")

# Cross-size: the same quantity for the thin column at N=96 (finding 25).
n96 = pd.read_csv(f"{SCRATCH}/spin_gnn/sudoku_n96_ff256_2026-10-27/"
                  f"depth_by_layers_n96_ff256.csv").query("rho_requested == @RHO")
print("\nN=96 thin column, well_formed_rate by cell:")
for tag in ("L08_ff00256", "L12_ff00256"):
    g = n96[n96.checkpoint.str.contains(tag)]
    if len(g):
        print(f"  {tag}: min {g.well_formed_rate.min():.4f}, max {g.well_formed_rate.max():.4f}")

fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.axhline(1.0, color="0.8", lw=1.0)
for name, tag, col, _ in CELLS:
    g = bins(tag)
    ax.plot(g["T"], g.well_formed_rate, "-o", color=col, ms=5, lw=1.8, label=f"N=24 {name}")
for tag, col in (("L08_ff00256", "#8c564b"), ("L12_ff00256", "#e377c2")):
    g = n96[n96.checkpoint.str.contains(tag)].sort_values("T")
    if len(g):
        ax.plot(g["T"], g.well_formed_rate, "--s", color=col, ms=5, lw=1.8,
                label=f"N=96 {tag}")
ax.set_xlabel("DD depth T"); ax.set_ylabel("well_formed_rate")
ax.set_title("Output grammar holds everywhere at N=24; it is N=96 that breaks")
ax.set_ylim(0.8, 1.02); ax.legend(fontsize=8, loc="lower left", ncol=2)
fig.tight_layout()

**`well_formed_rate` is 1.000 in all sixteen bins of the N=24 thin column**, including the
`L=2` cell that clears nothing. `truncated_rate` is 0.000 on the pool for every cell. So the
`L=2` failure is purely deductive: it emits perfectly well-formed traces that reach the wrong
answer, which is the opposite failure mode from N=96's `L=12/ff=256`.

That makes (d) a clean control. The grammar collapse is specific to the large-N deep-thin corner,
not a property of narrow MLPs in general, and the `L*` readings at N=24 and N=48 are measuring
deduction rather than output formatting.

It also sharpens the open question from finding 25: *why* does a 12-layer thin stack at N=96 lose
the grammar when a 6-layer thin stack at N=24 does not? Trainability is the leading hypothesis -
N=96 has the longer sequences and the deeper stack, and a loss of output structure early in
training is what learning-rate warmup exists to fix. Job **50008605** tests exactly that.

## 4. Pre-registrations, as resolved

Written into `scripts/slurm/sudoku2_n24_ff256_lstar.sbatch` with a named falsifier each, before
the job ran. That file is the pre-registration of record.

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | `L=3/ff=256` clears all 4 rho=0.60 bins, so `L*(24, ff=256) = 3` | **CONFIRMED** | clears 4 of 4 (0.978, 0.969, 0.951, 0.969); width factor 1.00x |
| (b) | on the n=1500 pool it is still resolvably **worse** than `L=3/ff=1024` | **FALSIFIED** | 0.9487 vs 0.9500, delta -0.0013, 50 won / 52 lost, p=0.92 |
| (c) | `L=2/ff=256` clears no bin and does not beat the wide `L=2` | **CONFIRMED**, reason refuted | clears 0 of 4; pooled 0.290 vs 0.385, Fisher p=0.001 |
| (d) | `well_formed_rate` stays >= 0.95 in both new cells | **CONFIRMED** | 1.000 in all 16 thin bins (N=96's worst was 0.853) |

**3 of 4.**

**(b) is the one that mattered and it is a negative result.** The prediction was that the width
cost would be present at N=24 but too small for a 4-bin gate to resolve, and that the 1500-instance
pool would expose it. The pool says there is nothing to expose: p=0.92 on 102 discordant instances
is a tie, not an underpowered test. At N=24 the 4x MLP narrowing is free on both the gate and the
pool, and the 1.5x width cost at N=48 and the >1.5x at N=96 have to be read against that clean zero.

**(c) held but its stated reason did not.** The sbatch header justified (c) by "width should matter
least at the small end". The data say width matters a great deal at the small end *below the knee*
(9.5 points at `L=2`) and not at all at or above it. The correct statement is not "width matters
less at small N" but "width substitutes for depth only while depth is insufficient".

**(a) and (d) are controls, not discoveries.** (a) measures a ladder floor that previous runs
assumed; (d) checks that finding 25's grammar collapse does not invalidate the gate everywhere.

## 5. Provenance, and what this does not settle

**Job.** `50007138` (`sud2n24ff256`), `-p sganguli --gres=gpu:1 --time=8:00:00`, **COMPLETED**
`ExitCode 0:0` in **00:34:11**. Script: `scripts/slurm/sudoku2_n24_ff256_lstar.sbatch`.
Log: `sud2n24ff256_50007138.out`.

**Commits - the run straddles two.** Jobs `cd "${SLURM_SUBMIT_DIR}"` and start a fresh `python -m`
per stage, so they import the live working tree and an edit mid-job changes which code later
stages run. That happened here:

| stage | commit | note |
|---|---|---|
| `L=3, ff=256` training | `e65b97e` | predates `warmup_epochs`; its `config.json` has no such key |
| `L=2, ff=256` training + both evals | `801c593` | adds `warmup_epochs`, **a no-op at the default 0** |

`801c593` only adds a `LinearLR` ramp that is skipped entirely when `warmup_epochs == 0` and
`lr_schedule == "constant"` (`_build_lr_schedule` returns `None`, exactly as before), so the two
cells are trained by identical code paths. The split is visible in the log at
`sud2n24ff256_50007138.out:328` and in the presence/absence of the `warmup_epochs` key in each
cell's `config.json`.

**Config.** `--n-nodes 24 --degree 5 --n-colors 4 --d-model 256 --nhead 4 --dim-feedforward 256
--n-train-instances 20000 --n-val-instances 300 --epochs 150 --lr 1e-3 --norm-first`,
`shuffle_adj=False`, `scratchpad` on, `max_len=210`. **Seed 0** (the sweep default) for both cells.
Depth eval: `--rho 0.75 0.60 --n-pool 600 --min-per-bin 40`. Pool:
`sudoku_agg_eval --n-instances 1500 --seed-start 50000000000`, wide `L=3` passed first.

**What this does not settle.**

- **The leak.** `shuffle_adj=False`, so every number above is a decoding score on the leaked-order
  task (finding 16). Finding 23 priced the leak-free ladder at 240000 instances x 100 epochs; it
  has not been run, and `scripts/slurm/sudoku2_leakfree_lstar.sbatch` still refuses to launch
  until `NTRAIN`/`EPOCHS`/`LAYERS` are set.
- **One seed per cell.** No seed variance anywhere in the grid.
- **Gate resolution at N=24.** Four bins, most near the ceiling. `L*=3` here is robust only
  because the gap between `L=2` and `L=3` is 0.68, not because the gate is sharp.
- **The width factor is not a paired comparison across N.** Each knee is measured on its own
  ladder at its own size, so the factor column in the headline table carries no p-value.

**Next, in order of information per GPU-hour.**

1. **Job 50008605 is already running** (`sud2n96warm`, `-p sganguli`, 28 h limit): `L=12` and
   `L=8` at `ff=256`, N=96, with a 5-epoch lr warmup as the *only* change from finding 25, `L=8`
   as the control. It decides whether `L*(96, ff=256) > 12` is about computation or about
   trainability - the one confound in the grid's most extreme cell.
2. `L=10/ff=1024` at N=96, which tightens the wide knee from "between 6 and 8" to one rung.
3. The leak-free `L*(T)` ladder at N=48 at 240000 x 100 epochs - the honest version of this whole
   grid, and the most expensive thing on the list.